In [1]:
import logging
logger = logging.basicConfig(level=logging.WARNING)

In [28]:
import accml.work_bench as wb
import accml.work_bench.all as wba
import accml.work_bench.lib_.custom.bessyii as b2
import accml.work_bench.custom.bluesky as wbb
import accml.work_bench.app.tune as wbt


In [29]:
from pathlib import Path
data_dir = Path(__name__).absolute().parent.parent.parent

In [30]:
from bluesky import RunEngine
from bluesky.callbacks import LiveTable
from databroker import catalog, catalog_search_path
from ophyd_async.core import soft_signal_rw

from accml.app.tune.tune_measurement import measure_tune_response
from accml.custom.bluesky.bluesky_measurement_execution_engine import (
    BlueskyMeasurementExecutionEngine,
)

In [31]:
# Setup informational signals (mock metadata)
info_sigs = {
    name: soft_signal_rw(str, name=name) for name in ["device_name", "channel_name"]
}
info_sigs["channel_value"] = soft_signal_rw(
    float, name="channel_value", precision=5
)

In [32]:
lt = LiveTable(
        [sig.name for _, sig in info_sigs.items()]
        + ["tune-transversal-x-sig", "tune-transversal-y-sig"]
        + ["tune-x", "tune-y"],
        # + list(actuators.values())
        default_prec=10,
    )

In [33]:
# from accml.custom.accml_lib.bessyii_on_tango.setup import setup



    

In [34]:
catalog_search_path()

('/home/mfp/.local/share/intake',
 '/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/share/intake')

In [35]:
RE = RunEngine()
[RE.subscribe(consumer) for consumer in [lt]]
# TODO: if you need to save the results, you need mongo and then uncomment the below two lines.
# db = catalog["heavy_local"]
# RE.subscribe(db.v1.insert)

[0]

In [36]:
yp, lm, _ = b2.bessyii_load_managers()

In [37]:
# Todo: remove this line after only Q3/Q4 are flagged as tune correction magnets
tune_correction_quads = [
    name for name in yp.tune_correction_quadrupole_names() if name[1] in ["3", "4"]
]
#  Find out to which power converters these are connected to
pc_names = {
    lm.forward(wba.LatticeElementPropertyID(name, "main_strength")).device_name: name
    for name in tune_correction_quads
}

In [44]:
devices=b2.bessyii_setup(prefix=None)
# devices=setup(prefix="")

In [45]:
state_cache = wba.StateCache(name="reference-data-cache-used-by-bluesky")

In [46]:
mexec=wbb.BlueskyMeasurementExecutionEngine(
        run_engine=RE,
        devices=devices,
        info_signals=info_sigs,
        cache=wba.StateCache(name="reference-data-cache-used-by-bluesky"),
    )
    

In [47]:
# Now I add a hack: I only use quadrupoles whoes power converter is unique
# I should rather work in device space right away
pc_names = list(set(pc_names))
await wbt.measure_tune_response(
    detectors=[wba.ReadCommand(id="tune", property="transversal")],
    quadrupole_pc_names=pc_names,
    mexec=mexec,
    measurement_values=[0, 1e0, 0, -1e0, 0],
    n_readings=3,
    retrieve_reference=True
)





+-----------+------------+-------------+--------------+---------------+------------------------+------------------------+
|   seq_num |       time | device_name | channel_name | channel_value | tune-transversal-x-sig | tune-transversal-y-sig |
+-----------+------------+-------------+--------------+---------------+------------------------+------------------------+
|         1 | 15:56:10.7 |     Q3P1T6R |  delta_set_c |       0.00000 |               1059.320 |                907.814 |
|         2 | 15:56:11.7 |     Q3P1T6R |  delta_set_c |       0.00000 |               1059.320 |                907.814 |
|         3 | 15:56:12.8 |     Q3P1T6R |  delta_set_c |       0.00000 |               1059.320 |                907.814 |
|         4 | 15:56:13.9 |     Q3P1T6R |  delta_set_c |       1.00000 |               1057.893 |                913.257 |
|         5 | 15:56:14.9 |     Q3P1T6R |  delta_set_c |       1.00000 |               1057.893 |                913.257 |
|         6 | 15:56:15

'7cbf4ca4-8bdd-4c48-8705-c167af014189'

CA.Client.Exception...............................................
    Context: "nbdt002:5064"
    Source File: modules/ca/src/client/tcpiiu.cpp line 925
    Current Time: Sat Jan 31 2026 16:18:19.823369041
..................................................................


In [ ]:
help(BlueskyMeasurementExecutionEngine)